In [16]:
# 加载环境变量
from dotenv import load_dotenv
from langchain_core.messages import AIMessage

load_dotenv()

True

# 定义模型
我们定义两个模型，一个多模态模型，一个聊天模型


In [17]:
from langchain.chat_models import init_chat_model
import os

# 多模态模型
multimodal_model = init_chat_model(
    model="qwen3.5-omni-plus",  # 模型名称，这里选择qwen3.5-plus，这是一个多模态模型，支持图片、文本、音频、视频
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

# 定义工具


In [18]:
from langchain_tavily import TavilySearch

# web搜索工具，使用tavily作为web搜索工具
web_search = TavilySearch(
    max_results=5,
    topic="general"
)

# 添加记忆管理

这里我们采用sqllite

In [19]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# 初始化checkpointer
checkpointer = SqliteSaver(sqlite3.connect("resources/personal_chief.db", check_same_thread=False))
# 自动建表
checkpointer.setup()

# 定义智能体

In [20]:
from langchain.agents import create_agent

system_prompt = """
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由，帮助用户快速做出决策。

请严格按照流程，优先调用 web_search 工具搜索食谱，再搜索不到的情况下才能自己发挥。
"""

agent = create_agent(
    model=multimodal_model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer
)

# 测试


In [27]:
from langchain.messages import HumanMessage

multimodal_message = HumanMessage(
    content=[
        {"type": "image",
         "url": "https://img95.699pic.com/photo/60066/5593.jpg_wh860.jpg"},
        {"type": "text", "text": "帮我看看这些食材能做些什么？"}
    ])

config = {"configurable": {"thread_id": "6"}}

In [28]:
response = agent.invoke({"messages": [multimodal_message]}, config)

In [29]:
# 友好打印
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'image', 'url': 'https://img95.699pic.com/photo/60066/5593.jpg_wh860.jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_ad6dcf8dedba46658a07b8f0)
 Call ID: call_ad6dcf8dedba46658a07b8f0
  Args:
    query: 金枪鱼牛排 牛肉 西兰花 牛油果 藜麦 食谱
    search_depth: advanced
================================= Tool Message =================================
Name: tavily_search

{"query": "金枪鱼牛排 牛肉 西兰花 牛油果 藜麦 食谱", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.reddit.com/r/mediterraneandiet/comments/1ai34uc/tuna_steak_with_quinoa_and_chopped_salad/?tl=zh-hans", "title": "金枪鱼牛排配藜麦和切碎的沙拉: r/mediterraneandiet", "content": "藜麦： 我按照包装上的说明准备的，但加了一个鸡汤块来增加风味。 金枪鱼： 我给金枪鱼牛排的两面都裹上了东西，然后用大火烤了1.5 分钟。", "score": 0.999928, "raw_content": null}, {"url": "https://www.reddit.com/r

In [30]:
response = agent.invoke(
    {"messages": [HumanMessage(content="我喜欢第3道菜，可以说的更详细点吗？")]},
    config
)

In [31]:
# 友好打印
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

好的，非常棒的选择！“缤纷能量碗”（Buddha Bowl）不仅颜值高，而且营养均衡，能让您一次性品尝到多种食材的风味和口感。

我已经为您检索并整合了详细的食谱方案，请查收这份专属您的烹饪指南。

---

### **🍽️ 私人厨师定制方案：缤纷能量碗 (Buddha Bowl)**

这道菜的精髓在于**“分门别类，色彩丰富”**。我们将所有食材精心烹制后，像艺术品一样码放在碗中，最后淋上灵魂酱汁。

#### **一、 食材准备与处理清单**

根据您的现有食材，我们这样分配：

*   **主食基底 (Base):**
    *   **三色藜麦:** 约半杯（煮熟后会膨胀）。它是完美的基底，富含蛋白质和纤维。
*   **核心蛋白质 (Protein - 二选一或组合):**
    *   **选项A (清爽海鲜): 香煎金枪鱼排。** 将1块金枪鱼排用盐和黑胡椒简单腌制。
    *   **选项B (浓郁肉香): 迷迭香蒜香牛肉粒。** 将牛肉切成2-3厘米的方块，用盐、黑胡椒、橄榄油和新鲜的迷迭香碎腌制。
*   **烤/焯蔬菜 (Vegetables):**
    *   **西兰花:** 切成小朵，用少许橄榄油、盐和黑胡椒拌匀。
    *   **彩椒:** 切成条状或块状，同样用橄榄油、盐和黑胡椒拌匀。
    *   **圣女果:** 可以对半切开，直接生食或稍微烤一下。
*   **新鲜点缀 (Fresh Toppings):**
    *   **牛油果:** 切片或切块，在组装前再处理以防氧化。
    *   **蓝莓:** 一小把，增加一抹亮色和清甜。
    *   **豆类:** 红豆需要提前浸泡并煮熟，作为额外的植物蛋白和膳食纤维来源。
*   **增香坚果 (Crunchy Elements):**
    *   **南瓜籽 & 葵花籽:** 可以稍微用平底锅干烤一下，直到微微金黄，香气四溢。
*   **灵魂酱汁 (The Dressing - 关键！):**
    *   **柠檬香草油醋汁:** 在一个小碗里，混合以下材料并充分搅打乳化：
 

# 流式调用

有工具调用的agent消息类型比较多，要正确的流式输出比较麻烦，这里给出一个示例


In [7]:
from langchain.messages import AIMessageChunk

for chunk, metadata in agent.stream(
            {"messages": [multimodal_message]},
            config,
            stream_mode="messages"
        ):
            if isinstance(chunk, AIMessageChunk) and chunk.content:
                print(chunk.content, end="", flush=True)

根据您提供的食材（土豆、猪肉、胡萝卜、番茄、青椒、茄子、大蒜、胡椒粒），我为您整理了一份营养均衡、制作简单的菜谱推荐清单。这些菜谱结合了您现有的食材，兼顾美味与健康，适合日常家庭烹饪。

---

### 🍽️ **推荐菜谱排名（按营养与制作难度综合评分）**

#### **1. 地三鲜（土豆、茄子、青椒）——经典家常菜（推荐指数：★★★★★）**
- **推荐理由**：这道菜是经典的“地三鲜”，搭配土豆、茄子和青椒，口感丰富，营养均衡。加入猪肉和大蒜，味道更香浓。
- **制作难度**：中等（约30分钟）
- **适合人群**：家庭聚餐、下饭神器
- **做法简述**：
  1. 土豆、茄子、青椒切块，胡萝卜切丝，猪肉切丁。
  2. 热锅加油，先煎土豆至金黄，盛出备用。
  3. 同一锅中炒香蒜末、姜片，加入猪肉丁翻炒至变色。
  4. 加入茄子、青椒、胡萝卜丝，翻炒至软。
  5. 加入生抽、盐、糖调味，最后加入土豆块，翻炒均匀即可。
  6. 可撒上胡椒粒或葱花点缀。

#### **2. 肉末茄子煲（推荐指数：★★★★☆）**
- **推荐理由**：茄子吸汁入味，搭配肉末和番茄，汤汁浓郁，非常适合配米饭。
- **制作难度**：中等（约40分钟）
- **适合人群**：喜欢浓郁口味的人
- **做法简述**：
  1. 茄子切块，用盐腌制10分钟，挤干水分。
  2. 热锅加油，将茄子煎至表面微焦，盛出备用。
  3. 锅中留底油，炒香蒜末、姜片，加入肉末翻炒至变色。
  4. 加入番茄块，炒至出汁，再加入茄子块、胡萝卜丝。
  5. 加入生抽、盐、糖调味，加适量水，小火炖煮15分钟。
  6. 最后撒上胡椒粒，收汁即可。

#### **3. 番茄胡萝卜炒土豆丝（推荐指数：★★★★）**
- **推荐理由**：这道菜色彩鲜艳，口感清爽，适合喜欢清淡口味的人。
- **制作难度**：低（约20分钟）
- **适合人群**：素食者、减脂人群
- **做法简述**：
  1. 土豆切丝，胡萝卜切丝，番茄切块。
  2. 热锅加油，先炒香蒜末，加入土豆丝翻炒至断生。
  3. 加入胡萝卜丝、番茄块，翻炒至番茄出汁。
  4. 加入盐、糖调味，撒上胡椒粒，翻炒均匀即可。

#### **4. 青椒炒肉片配土豆（推荐指数：★★★☆）**
- **推荐理由**：这道

# 获取会话历史

既然会话记忆交给了checkpointer管理，我们自然可以从中获取会话历史


In [7]:
checkpointer.get(config)

{'v': 4,
 'ts': '2026-03-05T04:32:04.748794+00:00',
 'id': '1f1184c4-318f-6c21-8005-8152a8ca695c',
 'channel_values': {'messages': [HumanMessage(content=[{'type': 'image', 'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}], additional_kwargs={}, response_metadata={}, id='a29981d9-1396-4397-9c42-8a3d4322a691'),
   AIMessage(content='', additional_kwargs={}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'qwen3-omni-flash', 'model_provider': 'openai'}, id='lc_run--019cbc43-db26-71e0-8385-79361516f7a7', tool_calls=[{'name': 'tavily_search', 'args': {'query': '用西兰花、三文鱼、鸡胸肉、蘑菇、小番茄、红椒、生菜、白萝卜做的菜谱', 'search_depth': 'advanced'}, 'id': 'call_3971f410f521440fab0430', 'type': 'tool_call'}], invalid_tool_calls=[]),
   ToolMessage(content='{"query": "用西兰花、三文鱼、鸡胸肉、蘑菇、小番茄、红椒、生菜、白萝卜做的菜谱", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://cookpad.co

In [71]:
for m in checkpointer.get(config)['channel_values']['messages']:
    print(type(m))
    print(m)

<class 'langchain_core.messages.human.HumanMessage'>
content=[{'type': 'image', 'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}] additional_kwargs={} response_metadata={} id='8c0219c3-2c70-46bb-bfa2-400a516e0117'
<class 'langchain_core.messages.ai.AIMessage'>
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 2151, 'total_tokens': 2206, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': None, 'rejected_prediction_tokens': None, 'text_tokens': 55}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'image_tokens': 262, 'text_tokens': 1889}}, 'model_provider': 'openai', 'model_name': 'qwen3-omni-flash', 'system_fingerprint': None, 'id': 'chatcmpl-a5c8f901-eaa4-9903-96e4-334a9cb802da', 'finish_reason': 'tool_calls', 'logprobs': None} id='l

定义一个便捷的获取会话历史的函数：


In [78]:
def get_messages(thread_id: str) -> list[dict[str, str]]:
    """获取会话历史"""

    # 根据 thread_id 查询 checkpoint
    cp = checkpointer.get({"configurable": {"thread_id": thread_id}})

    # 如果不存在，返回空列表
    if not cp:
        return []

    # 安全获取 messages
    channel_values = cp.get("channel_values")
    if not channel_values:
        return []

    messages = channel_values.get("messages", [])
    if not messages:
        return []

    # 转换消息格式
    result = []
    for msg in messages:
        if not msg.content:
            continue

        if isinstance(msg, HumanMessage):
            result.append({"role": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            result.append({"role": "assistant", "content": msg.content})

    return result

In [82]:
get_messages("1")

[]

# 清空会话历史


In [81]:
checkpointer.delete_thread("1")